In [2]:

import os, io, time, json, hashlib, pathlib, sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import re
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from datetime import datetime, timedelta
from config import *
# from functions1 import *
from functions2 import *

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 725,
    max_age: int = 24,
    no_fx: bool = False,
    usd_shift: bool = False,
    DEBUG: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.
        holdings: list of dicts with tickers:
      - name: row/column label in outputs
      - ticker: EODHD tickerbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)
      - risk_fx: bool; when False, strip FX volatility (i.e., hedged in risk). When True or missing, include FX volatility.

        Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """
    params = {  'from': START, 
                'to': today,
                'api_token': EOD_API    }
    url_fx = f'https://eodhd.com/api/eod/'

    def norm_risk_fx(val) -> bool:
        """
        Normalize various user inputs to a boolean with this convention:
        - False => strip FX volatility (hedged in risk)
        - True  => include FX volatility (unhedged in risk)
        Defaults to True when missing/unknown.
        """
        if isinstance(val, bool):
            return val
        if val is None:
            return True
        if isinstance(val, str):
            v = val.strip().lower()
            if v in ("none", "hedged", "no", "false", "0", "off"):
                return False
            if v in ("true", "yes", "1", "on", "unhedged"):
                return True
        if isinstance(val, (int, float)):
            return bool(val)
        return True
    
    fx_map = make_fx_map(url_fx, holdings, params, max_age, no_fx, usd_shift)
    if DEBUG:
        print(f">>>FX keys loaded: {list(fx_map.keys())}")
    print('-------------fetching assets-------------')

    url = f'https://eodhd.com/api/eod/'

    assets_close_local_df = pd.DataFrame()
    assets_close_chf_df = pd.DataFrame()
    # BUILD CHF CLOSE SERIES PER ASSET
    for h in holdings:
        name = h['name']
        asset_close_local_s = fetch_asset_series(url, h, fx_map, params=params, max_age=max_age, lookback_days=lookback_days)
        assets_close_local_df[name] = asset_close_local_s
        ccy = h.get('ccy','').upper()
        gbx = bool(h.get('gbx', False))
        series_local = asset_close_local_s / 100.0 if gbx else asset_close_local_s
        # Decide whether to strip FX vol in returns (risk_fx=False)
        risk_fx_bool = norm_risk_fx(h.get('risk_fx', True))
        strip_fx = not risk_fx_bool
        if ccy == 'CHF' or no_fx or strip_fx or h.get('type','').lower()=='cash':
            asset_close_chf_s = series_local.rename(name)
        else:
            fx = fx_map.get(f"{ccy}CHF", pd.Series(dtype=float))
            before = fx.reindex(series_local.index)
            fx_aligned = before.ffill()
            asset_close_chf_s = (series_local * fx_aligned).dropna().rename(name)
        assets_close_chf_df[name] = asset_close_chf_s
   
    # Hedged cash lines: risk_fx=False → flatten to 1.0 to remove FX vol
    hedged_cash = [
        h['name'] for h in holdings
        if h.get('type','').lower() == 'cash' and (not norm_risk_fx(h.get('risk_fx', True)))
    ]
    for n in hedged_cash:
        if n in assets_close_chf_df.columns:
            assets_close_chf_df[n] = 1.0


    # ALIGN ON COMMON DATES AND RESTRICT TO LOOKBACK WINDOW
    prices_df = pd.DataFrame(assets_close_chf_df).dropna()
   
    if DEBUG and not prices_df.empty:
        print(
            f">>>Common window: {prices_df.index.min().date()} → {prices_df.index.max().date()} "
            f"({len(prices_df)} rows before tail)"
        )

    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.73):
        print(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
        )
    
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")

    asof = prices_df.index[-1]
    chf_values = {}

    # GET CHF VALUE FOR EACH HOLDING (valuation always in CHF at as-of)
    for h in holdings:
        name = h['name']
        size = h.get('position', 0.0)
        chf_value = get_holding_value_chf(h, fx_map, assets_close_local_df, assets_close_chf_df, asof)
        # print(f'CHF value {size} of {h["name"]}: {chf_value:.2f}')
        if chf_value is not None:
            chf_values[name] = chf_value
        
    total_val = sum(chf_values.values())
    print(f'LOOKBACK DAYS/REGIME: {lookback_days}')
    print(f"Total portfolio value (CHF): {total_val:.2f}")

    # CALCULATE WEIGHTS (CHF)
    weights = pd.Series()
    for h in holdings:
        name = h["name"]
        size = h.get('position', 0.0)
        value = float(chf_values[name])
        weight = value / total_val
        weights[name] = weight

        # JUST FOR THE PRINTING
        last = assets_close_chf_df[name].iloc[-1]
        print(f"    {name}: value {value:.2f} CHF,  last: {last:.2f} * {size}")


    if not np.isclose(weights.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {weights.sum():.6f}" "check postions input in holdings.")
    return rets_df, prices_df, weights



def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [3]:
computershare = [
    {"name":"Unilever", "ticker":"ULVR.LON", "ccy":"GBP", "gbx":True,  "value_chf": 25000},
    {"name":"Shell",    "ticker":"SHEL.LON", "ccy":"GBP", "gbx":True,  "value_chf": 13000},
    {"name":"NatWest",  "ticker":"NWG.LON",  "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Barclays", "ticker":"BARC.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Tesco",    "ticker":"TSCO.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"SWDA",     "ticker":"SWDA.LON", "ccy":"GBP", "gbx":True,  "value_chf":  12000},
    {"name":"EMIM",     "ticker":"EMIM.LON", "ccy":"GBP", "gbx":True,  "value_chf":  8000},
    {"name":"IBM",      "ticker":"IBM",      "ccy":"USD", "gbx":False, "value_chf":  4000},
    {"name":"ERNS",     "ticker":"ERNS.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
]
IBKR_real = [
    {"name":"EMIM",     "ticker":"EMIM.LSE", "ccy":"GBP", "gbx":True, "position": 200},
    {"name":"ERNS",     "ticker":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 89, 'risk_fx': False},
    {"name":"HEAL",     "ticker":"HEAL.LSE", "ccy":"GBP", "gbx":False, "position": 180},
    {"name":"IBM",     "ticker":"IBM.US", "ccy":"USD", "gbx":False, "position": 9},

    {"name":"SGLN",      "ticker":"SGLN.LSE",      "ccy":"GBP", "gbx":True, "position": 42},

    {"name":"VUAG",     "ticker":"VUAG.LSE", "ccy":"GBP","USD_exposure": 1, "gbx":False, "position": 28},
    {"name":"WSML",     "ticker":"WSML.LSE", "ccy":"USD","USD_exposure": 0.58, "gbx":False, "position": 483},
    {"name": "SIKA", "ticker": "SIKA.SW", "ccy": "CHF", "gbx":False, "position": -9},
    {"name": "YCA", "ticker": "YCA.LSE", "ccy": "GBP", "gbx":True, "position": 178},
    {"name":"VEU",     "ticker":"VEU.US", "ccy":"USD", "gbx":False, "position": 68,},

    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 9370},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": -2800, "risk_fx": True },
    {"name": "CASH_USD", "type": "cash", "ccy": "USD", "amount": -3560},
]
IBKR_test =[
    {"name":"EMIM",     "ticker":"EMIM.LSE", "ccy":"GBP", "gbx":True, "position": 147},
    # {"name":"ERNS",     "ticker":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 89, 'risk_fx': False},
    {"name":"HEAL",     "ticker":"HEAL.LSE", "ccy":"GBP", "gbx":False, "position": 180},
    {"name":"IBM",     "ticker":"IBM.US", "ccy":"USD", "gbx":False, "position": 9},

    {"name":"SGLN",      "ticker":"SGLN.LSE",      "ccy":"GBP", "gbx":True, "position": 42},

    {"name":"VUAG",     "ticker":"VUAG.LSE", "ccy":"GBP","USD_exposure": 1, "gbx":False, "position": 37},
    {"name":"WSML",     "ticker":"WSML.LSE", "ccy":"USD","USD_exposure": 0.58, "gbx":False, "position": 483},
    {"name": "SIKA", "ticker": "SIKA.SW", "ccy": "CHF", "gbx":False, "position": -9},
    {"name": "YCA", "ticker": "YCA.LSE", "ccy": "GBP", "gbx":True, "position": 178},
    {"name":"VEU",     "ticker":"VEU.US", "ccy":"USD", "gbx":False, "position": 78,},

    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 9370},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": -2800, "risk_fx": True },
    {"name": "CASH_USD", "type": "cash", "ccy": "USD", "amount": -3560},
]
holdings = IBKR_real
MAX_AGE = 12
LOOKBACK_DAYS = 756
DEBUG = False
START = '2020-01-01'

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=LOOKBACK_DAYS, max_age=MAX_AGE, no_fx=False, usd_shift=False, DEBUG=DEBUG)

risk = portfolio_risk(rets_df, w)

# print( rets_df.tail(3) )

print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print(" CORE EQUITY BAND = 15 - 30%, DIVERSIFIERS = 5 - 12%")
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
print(f'COVARIANCE:')
print(f'{risk["cov_annual"]}')

-------------fetching currencies-------------
-------------fetching assets-------------
LOOKBACK DAYS/REGIME: 756
Total portfolio value (CHF): 35612.31
    EMIM: value 6875.60 CHF,  last: 34.38 * 200
    ERNS: value 9688.07 CHF,  last: 101.80 * 89
    HEAL: value 1551.82 CHF,  last: 8.62 * 180
    IBM: value 1914.66 CHF,  last: 212.74 * 9
    SGLN: value 2429.66 CHF,  last: 57.85 * 42
    VUAG: value 2837.45 CHF,  last: 101.34 * 28
    WSML: value 3376.08 CHF,  last: 6.99 * 483
    SIKA: value -1564.65 CHF,  last: 173.85 * -9
    YCA: value 1125.83 CHF,  last: 6.32 * 178
    VEU: value 3832.74 CHF,  last: 56.36 * 68
    CASH_CHF: value 9370.00 CHF,  last: 1.00 * 0.0
    CASH_GBP: value -2994.04 CHF,  last: 1.07 * 0.0
    CASH_USD: value -2830.91 CHF,  last: 0.80 * 0.0
Portfolio σ (annualized, CHF): 7.77%
 CORE EQUITY BAND = 15 - 30%, DIVERSIFIERS = 5 - 12%
          Weight  Vol_1Y_CHF    MRC  PRC_%
EMIM       0.193       0.160  0.141   35.1
WSML       0.095       0.184  0.147   18.0
VE

In [30]:
def eod_search(quey: str, token: str):
    import requests, pandas as pd
    url = f"https://eodhd.com/api/search/{quey}?api_token={token}&fmt=json"
    r = requests.get(url, timeout=30); r.raise_for_status()
    hits = r.json()
    # Return a small table to pick from
    return pd.DataFrame([{
        "code": h.get("Code"),
        "exchange": h.get("Exchange"),
        "name": h.get("Name"),
        "currency": h.get("Currency"),
        "type": h.get("Type"),
        "startdate": h.get("StartDate"),
        # earliet date

    } for h in hits])

# Usage:
df = eod_search("heal", EOD_API)
# pick the line with the longest available history (often XETRA/LSE/SIX)
print(df)

          code exchange                                               name  \
0         HEAL       JK                              Medikaloka Hermina PT   
1         HEAL      LSE  iShares Healthcare Innovation UCITS ETF USD (Acc)   
2         HEAL       US                            Global X HealthTech ETF   
3         HEAL       AS  iShares Healthcare Innovation UCITS ETF USD (Acc)   
4         HEAL       SW  iShares Healthcare Innovation UCITS ETF USD (Acc)   
5   HEALTHIETF      NSE                                         HEALTHIETF   
6       HEALTH       HE                           Nightingale Health Oyj B   
7      HEALTHY      NSE         Aditya Birla Sun Life Nifty Healthcare ETF   
8       HEALTH       BK                            Health Empire Corp. PCL   
9    HEALTHADD      NSE                                          HEALTHADD   
10      000991      SHG                    CSI All Share Health Care Index   
11         XLV       US               Health Care Select Sector 

## GET THE EARLIEST DATE ##

In [ ]:
START = '2020-01-02'
ticker = f'URNM.LSE'
params = {
    'from': START, 
    'to': today,
    'api_token': EOD_API    
}
url_fx = f'https://eodhd.com/api/eod/{ticker}'

# Fetch EODHD daily FX and build a Series
fx_df = fetch_csv_robust(url_fx, params=params, ticker=ticker, max_age=24)

arse
VEU.US - using cached data


## Equity correlation drift check
We’ll:
- compute a 60-day rolling average pairwise correlation across the equity ETFs in your `rets_df`.
- show the 1-year view you’re using now (limited by `XMWX`).
- also compute a 3-year view excluding `XMWX` (if data exist), to see whether the increase is structural or just recent.

In [ ]:
# 1) Current 1-year view (already in rets_df)
# Pick equity ETFs present in rets_df
all_cols = list(rets_df.columns)
EQUITY_LIKE = [c for c in ["EMIM","VUAG","WSML","XMWX","IBM"] if c in all_cols]

if len(EQUITY_LIKE) >= 2:
    rets_eq_1y = rets_df[EQUITY_LIKE]
    # 60D rolling average of pairwise correlation (off-diagonal mean)
    def offdiag_mean(corr_m):
        if corr_m.shape[0] < 2:
            return np.nan
        n = corr_m.shape[0]
        return (corr_m.values.sum() - n) / (n*(n-1))

    roll_avg_corr_1y = (
        rets_eq_1y.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
    )

    print("1Y rolling(60) avg pairwise equity corr (tail):")
    print(roll_avg_corr_1y.dropna().tail(10))
else:
    print("Not enough equity-like tickers to compute 1Y pairwise correlation.")

# 2) Longer 3Y view excluding XMWX (if data available): re-fetch or extend window is out of scope here,
# but we can approximate by checking if older data exist in prices_df; if not, we demonstrate exclusion-only.
try:
    # If you want to explicitly exclude XMWX to avoid its shorter history limiting the window
    EQUITY_NO_XMWX = [c for c in EQUITY_LIKE if c != "XMWX"]
    if len(EQUITY_NO_XMWX) >= 2:
        # Use available rets_df (1Y). For a true 3Y view, rerun build_returns_matrix_in_chf with lookback_days=756.
        rets_eq_ex = rets_df[EQUITY_NO_XMWX]
        roll_avg_corr_ex = (
            rets_eq_ex.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
        )
        print("Ex-XMWX rolling(60) avg pairwise equity corr (tail):")
        print(roll_avg_corr_ex.dropna().tail(756))
    else:
        print("Not enough non-XMWX equity-like tickers to compute ex-XMWX correlation.")
except Exception as e:
    print("Correlation analysis note:", e)


1Y rolling(60) avg pairwise equity corr (tail):
Date
2025-09-02    0.549627
2025-09-03    0.551725
2025-09-04    0.549764
2025-09-05    0.545939
2025-09-08    0.545741
2025-09-09    0.543770
2025-09-10    0.529330
2025-09-11    0.523707
2025-09-12    0.510802
2025-09-15    0.515767
dtype: float64
Ex-XMWX rolling(60) avg pairwise equity corr (tail):
Date
2024-12-02    0.426043
2024-12-03    0.410413
2024-12-04    0.410285
2024-12-05    0.417191
2024-12-06    0.403619
                ...   
2025-09-09    0.506077
2025-09-10    0.487784
2025-09-11    0.483393
2025-09-12    0.470275
2025-09-15    0.477077
Length: 192, dtype: float64


## 3-year view excluding XMWX
We will rebuild returns with a longer lookback (756 trading days ≈ 3y) and exclude `XMWX` so the window isn’t truncated, then recompute the 60D rolling average pairwise correlation.

## USD exposure check (manual)
This cell computes a per-holding USD exposure using explicit `USD_exposure` look-through where provided (e.g., WSML≈0.58), defaults to 1.0 for USD-priced assets not marked `risk_fx=False` (i.e., unhedged in risk), and includes `CASH_USD` as a negative hedge. Values are in CHF; a hedge recommendation in USD is derived from the `CASH_USD` series (USDCHF).

## Period-based volatility view (e.g., last 6 months)
Use this section to compute volatility and risk contributions over a specific recent window, and compare with the immediately prior, equal-length window. Set `MONTHS` (default 6). If your `rets_df` covers more than this window, the slice will use calendar months based on the latest date in `rets_df`.

In [ ]:
# Parameters
MONTHS = 6  # set to any integer number of months for the analysis window
RECENT_START = None  # Optional explicit start date string like "2025-03-01" (your liberation day). If set, overrides MONTHS.
ANNUALIZATION = 252.0
if MONTHS is not None and (not isinstance(MONTHS, int) or MONTHS <= 0):
    raise ValueError("MONTHS must be a positive integer when used.")

# Determine date boundaries
latest = rets_df.index.max()
first = rets_df.index.min()
if pd.isna(latest):
    raise ValueError("rets_df is empty or has invalid index.")

if RECENT_START:
    # Use explicit recent start date (e.g., liberation day)
    recent_start = pd.Timestamp(RECENT_START)
    if recent_start > latest:
        raise ValueError(f"RECENT_START {recent_start.date()} is after latest return date {latest.date()}.")
    recent_start = max(recent_start.normalize(), first)
    # Make prior window equal length in time
    window_len = latest.normalize() - recent_start
    prior_end = recent_start
    prior_start = max((prior_end - window_len).normalize(), first)
else:
    # Use calendar-month-based window
    recent_start = (latest - pd.DateOffset(months=MONTHS)).normalize()
    prior_start  = (latest - pd.DateOffset(months=2*MONTHS)).normalize()
    recent_start = max(recent_start, first)
    prior_start  = max(prior_start, first)
    prior_end    = recent_start

# Build slices: recent window and prior equal-length window
rets_recent = rets_df.loc[rets_df.index >= recent_start]
rets_prior  = rets_df.loc[(rets_df.index >= prior_start) & (rets_df.index < recent_start)]

def describe_period(df: pd.DataFrame, label: str):
    if df.empty or df.shape[0] < 20:
        print(f"[{label}] Window too short: {df.shape[0]} rows. Skipping.")
        return None
    # Reuse portfolio_risk with the existing weights w (valued at the global as-of).
    res = portfolio_risk(df, w)
    print(f"[{label}] rows={df.shape[0]}, from {df.index.min().date()} to {df.index.max().date()}")
    print(f"[{label}] Portfolio σ (annualized, CHF): {res['port_vol']:.2%}")
    # Show top contributors
    print(res["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1}).head(15))
    return res

res_recent = describe_period(rets_recent, f"Recent {MONTHS}M" if not RECENT_START else f"Recent since {pd.Timestamp(RECENT_START).date()}")
res_prior  = describe_period(rets_prior,  f"Prior window")

# Compare if both exist
if (res_recent is not None) and (res_prior is not None):
    # Asset vol deltas (annualized)
    vol_recent = res_recent["summary"]["Vol_1Y_CHF"]
    vol_prior  = res_prior["summary"]["Vol_1Y_CHF"].reindex(vol_recent.index)
    vol_delta  = (vol_recent - vol_prior).dropna().sort_values(ascending=False)
    print("\nChange in asset vols (annualized, CHF): top movers")
    print(vol_delta.round(3).head(15))

    # Percent risk contribution deltas
    prc_recent = res_recent["summary"]["PRC_%"]
    prc_prior  = res_prior["summary"]["PRC_%"].reindex(prc_recent.index)
    prc_delta  = (prc_recent - prc_prior).dropna().sort_values(ascending=False)
    print("\nChange in PRC (percentage points): top movers")
    print(prc_delta.round(2).head(15))

    # Portfolio vol delta
    port_vol_delta = res_recent["port_vol"] - res_prior["port_vol"]
    print(f"\nChange in portfolio σ (annualized, CHF): {port_vol_delta:+.2%}")